# DeepRL Monopoly — Builder+DealMaker self-play (no ASU distillation)

Trains a DDQN/PPO agent via pure self-play RL against `TheBuilder` + `TheDealMaker`
(the two strongest fixed-policy opponents). Reward includes a custom
`liquidity_risk` shaping term (see `monopoly_game_engine/env.py`).

ASU is not used anywhere in this notebook — no teacher, no labels, no distillation.

## 1. Mount Drive (for checkpoints) — optional, skip if you don't need persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepRL_Monopoly_ckpt'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Clone the repo (feature/asu-teacher-distillation branch)

In [ ]:
%cd /content
!rm -rf DeepRL_Monopoly
!git clone --branch feature/asu-teacher-distillation https://github.com/EnzeCbe/monopoly-boom.git DeepRL_Monopoly
%cd DeepRL_Monopoly

## 3. Check GPU + torch (Colab ships CUDA-enabled torch already)

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('No GPU — Runtime > Change runtime type > GPU (T4), then re-run this cell.')

## 3.5. Quick timing test (100 games) — run this first

Same measurement we ran locally (CPU-only: ~180s/game once the replay buffer
fills and gradient updates kick in). This tells us the real GPU speedup before
committing to a multi-thousand-game run.

In [ ]:
import sys, time
sys.path.insert(0, '.')
from monopoly_game_engine.train import train
from monopoly_game_engine.agent_ddqn import DDQNAgent

agent = DDQNAgent(player_id=0, hybrid=True, device='auto')
print('agent device:', agent.device)

t0 = time.time()
history = train(agent, is_ppo=False, hybrid=True, n_games=100, log_every=10, seed=1)
elapsed = time.time() - t0
print('ELAPSED_SECONDS', elapsed)
print('sec/game', elapsed / 100)
print('local CPU baseline was ~180 sec/game -> speedup:', 180 / (elapsed / 100))

## 4. Train

`--device auto` picks CUDA automatically. `--checkpoint-every` saves a resumable
checkpoint (format 3: optimizer state + replay buffer + progress) so a disconnected
Colab session can `--resume` from the same `--out` path.

In [ ]:
import os
OUT = f"{CHECKPOINT_DIR}/ddqn_builder_dealmaker.pt" if 'CHECKPOINT_DIR' in dir() else 'artifacts/ddqn_builder_dealmaker.pt'
os.makedirs(os.path.dirname(OUT), exist_ok=True)

!PYTHONIOENCODING=utf-8 python tools/train_and_save.py \
  --algo ddqn --games 5000 --device auto \
  --checkpoint-every 100 \
  --out "{OUT}"

## 5. Resume after a disconnect (same --out path, --resume flag)

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/train_and_save.py \
  --algo ddqn --games 20000 --device auto --seed 42 --resume \
  --checkpoint-every 100 \
  --out "{OUT}"

## 6. Quick eval against Builder + DealMaker after training

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/play_game.py \
  --algo ddqn --players 4 --model "{OUT}"